# Praktikum 2 - Deployment ke Hugging Face (Docker + Flask)

**Nama:** Agus
**NIM:** 244107020049

[Sumber modul](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js15-pipeline-machine-learning-and-deployment/praktikum-2)

Praktikum 2 modul berupa berkas, bukan sel notebook. Berkas Langkah 1-4 sudah disiapkan di folder [`daynight-project/`](daynight-project/):

| Berkas | Isi |
| --- | --- |
| `app.py` | Kode Flask modul (Langkah 2), tidak diubah |
| `requirements.txt` | Daftar library modul (Langkah 3) dengan versi dikunci sama seperti saat training |
| `Dockerfile` | Konfigurasi modul (Langkah 4), dengan base image `python:3.12-slim` |
| `day_night_model.h5`, `scaler.pkl` | Artefak hasil Praktikum 1 |
| `README.md` | Header konfigurasi Space (`sdk: docker`, `app_port: 7860`) dan langkah upload |

**Alasan perubahan versi.** `tensorflow-cpu` 2.21 membutuhkan Python ≥ 3.10, sehingga `python:3.9-slim` dari modul tidak dapat memasangnya.
Model `.h5` disimpan dengan Keras 3.15 dan `scaler.pkl` dengan scikit-learn 1.9.1. Versi yang berbeda dapat gagal memuat model atau memberi
peringatan *InconsistentVersionWarning* saat membuka pickle. Karena itu versi di `requirements.txt` dikunci sama dengan environment training.

Notebook ini menguji Langkah 7 secara lokal sebelum upload: aplikasi Flask dijalankan dengan *test client*, lalu citra test diunggah ke `/predict`.
Langkah 5-6 (membuat Space, token, dan `git push`) memerlukan akun Hugging Face pribadi, sehingga dijelaskan di akhir notebook dan di `daynight-project/README.md`.

In [1]:
import sys
import numpy as np
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
print("Kernel:", sys.executable)
print("TensorFlow:", tf.__version__)

Kernel: /Users/jergal/College/machine-learning-smt-5/.venv/bin/python
TensorFlow: 2.21.0


In [2]:
from pathlib import Path
for f in sorted(Path('daynight-project').iterdir()):
    print(f'{f.name:22s} {f.stat().st_size / 1e6:7.2f} MB')
print('\n--- requirements.txt ---'); print(Path('daynight-project/requirements.txt').read_text())
print('--- Dockerfile ---'); print(Path('daynight-project/Dockerfile').read_text())

Dockerfile                0.00 MB
README.md                 0.00 MB
app.py                    0.00 MB
day_night_model.h5       17.77 MB
requirements.txt          0.00 MB
scaler.pkl                0.83 MB

--- requirements.txt ---
flask==3.1.3
tensorflow-cpu==2.21.0
numpy==2.5.5
scikit-learn==1.9.1
scikit-image==0.26.0
opencv-python-headless==5.0.0.93
gunicorn==27.0.1

--- Dockerfile ---
# Base Image Python 3.12 Slim (modul: 3.9; tensorflow-cpu 2.21 membutuhkan Python >= 3.10)
FROM python:3.12-slim

# Install Library Sistem untuk OpenCV (Debian Bookworm/Trixie Compatible)
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan HF)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File 

## Uji aplikasi Flask secara lokal (Langkah 7, sebelum upload)

In [3]:
import os, re, sys, importlib
os.chdir('daynight-project')          # app.py memuat model/scaler dengan path relatif, sama seperti di container
sys.path.insert(0, os.getcwd())
app_module = importlib.import_module('app')
client = app_module.app.test_client()

home = client.get('/')
print('GET / ->', home.status_code, '| form upload:', b'type="file"' in home.data)

✅ System Loaded Successfully
GET / -> 200 | form upload: True


In [4]:
import pandas as pd
rows = []
for label in ['day', 'night']:
    for img_path in sorted(Path('../data/images/test', label).glob('*.jpg')):
        with open(img_path, 'rb') as fh:
            resp = client.post('/predict', data={'file': (fh, img_path.name)}, content_type='multipart/form-data')
        text = resp.get_data(as_text=True)
        hasil = re.search(r'Hasil: ([^<]+)', text)
        rows.append({'file': img_path.name, 'label': label, 'status': resp.status_code,
                     'prediksi': hasil.group(1) if hasil else text[:80]})
os.chdir('..')
res = pd.DataFrame(rows)
res['benar'] = [p.startswith('Day' if l == 'day' else 'Night') for p, l in zip(res['prediksi'], res['label'])]
print('Semua status 200:', (res['status'] == 200).all())
print(f"Akurasi lewat endpoint /predict: {res['benar'].mean():.4f} ({res['benar'].sum()}/{len(res)})")
res.groupby(['label', 'prediksi']).size().unstack(fill_value=0)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


... (312 outputs left out)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


Semua status 200: True
Akurasi lewat endpoint /predict: 0.9062 (145/160)


prediksi,Day (Siang),Night (Malam)
label,,
day,75,5
night,10,70


Akurasi endpoint dibandingkan dengan akurasi test di Praktikum 1. Jika preprocessing di `app.py` berbeda dengan saat training
(*training-serving skew*, Sculley dkk., 2015), angkanya akan berbeda. Training memakai `RGB → resize → RGB2GRAY`, sedangkan `app.py`
memakai `BGR (imdecode) → resize → BGR2GRAY`. Keduanya menghasilkan citra grayscale yang sama karena bobot luminans
(0,299 R + 0,587 G + 0,114 B) diterapkan pada kanal yang sesuai.

## Langkah 5-7: Upload ke Hugging Face Spaces

Jalankan dari terminal (Git dan Git LFS terpasang). Ganti `USERNAME_ANDA`:

```bash
# Langkah 5: buat Space baru di https://huggingface.co/new-space
#   Space name: daynight-classifier-244107020049 | SDK: Docker | Template: kosong | Visibility: Public
#   Settings > Access Tokens > Create new token (Write)

# Langkah 6: clone, salin berkas, push
git lfs install
git clone https://huggingface.co/spaces/USERNAME_ANDA/daynight-classifier-244107020049
cp JS15/daynight-project/{app.py,Dockerfile,requirements.txt,day_night_model.h5,scaler.pkl,README.md} daynight-classifier-244107020049/
cd daynight-classifier-244107020049
git add .
git commit -m "Deploy aplikasi ke HF"
git push      # Username: USERNAME_ANDA, Password: access token (bukan password akun)
```

`day_night_model.h5` berukuran sekitar 18 MB. Hugging Face menolak file > 10 MB yang tidak memakai Git LFS. Template Space sudah memuat
`.gitattributes` yang melacak `*.h5` dan `*.pkl` dengan LFS, sehingga cukup pastikan `git lfs install` sudah dijalankan. Langkah 7:
buka tab **App**, tunggu status *Building* menjadi *Running*, lalu unggah citra siang/malam.

## Rujukan

1. Merkel, D. (2014). Docker: Lightweight Linux containers for consistent development and deployment. *Linux Journal*, 2014(239), Article 2.
2. Sculley, D., dkk. (2015). Hidden technical debt in machine learning systems. *Advances in Neural Information Processing Systems 28*, 2503-2511.
3. Hugging Face. *Docker Spaces* (dokumentasi). https://huggingface.co/docs/hub/spaces-sdks-docker